In [14]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

In [15]:
BASE_DIR = '../Fruits_data/train'
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 30  # Slightly higher for fine-tuning
MODEL_PATH = '../MLMODELS/fruits_veg_identify.keras'

In [16]:
for cls in os.listdir(BASE_DIR):
    n_images = len(os.listdir(os.path.join(BASE_DIR, cls)))
    print(f"{cls}: {n_images} images")

apple: 68 images
banana: 75 images
beetroot: 88 images
bell pepper: 90 images
cabbage: 92 images
capsicum: 89 images
carrot: 82 images
cauliflower: 79 images
chilli pepper: 87 images
corn: 87 images
cucumber: 94 images
eggplant: 84 images
garlic: 92 images
ginger: 68 images
grapes: 100 images
jalepeno: 88 images
kiwi: 88 images
lemon: 82 images
lettuce: 97 images
mango: 86 images
onion: 94 images
orange: 69 images
paprika: 83 images
pear: 89 images
peas: 100 images
pineapple: 99 images
pomegranate: 79 images
potato: 77 images
raddish: 81 images
soy beans: 97 images
spinach: 97 images
sweetcorn: 91 images
sweetpotato: 69 images
tomato: 92 images
turnip: 98 images
watermelon: 84 images


In [17]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.1,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range =[0.7,1.3],
    fill_mode='nearest'
)

In [18]:
train_generator = datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='training',
    class_mode='categorical'
)

validation_generator = datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    class_mode='categorical'
)

Found 2823 images belonging to 36 classes.
Found 292 images belonging to 36 classes.


In [19]:
# Save class labels
labels = '\n'.join(sorted(train_generator.class_indices.keys()))
with open('labels.txt', 'w') as f:
    f.write(labels)

In [20]:
y = train_generator.classes
classes = np.unique(y)
class_weights = compute_class_weight('balanced', classes=classes, y=y)
class_weights = dict(enumerate(class_weights))

In [21]:
IMAGE_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SHAPE,
    include_top=False,
    weights='imagenet'
)
# Freeze all layers except last 30 for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

In [22]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_generator.class_indices), activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 36)             │         9,252 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,595,172 (9.90 MB)

 Trainable params: 2,192,292 (8.36 MB)

 Non-trainable params: 402,880 (1.54 MB)

In [23]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

In [24]:
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [25]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True)
]

In [26]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

c:\Users\gcbho\OneDrive\Desktop\CAPSTONE\zipcartprototype\ZipCartPrototype\backend\pythonservices\.venv\Lib\site-packages\PIL\Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.1385 - loss: 3.3315 - val_accuracy: 0.3767 - val_loss: 2.3733
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 137s 3s/step - accuracy: 0.4070 - loss: 2.2514 - val_accuracy: 0.5616 - val_loss: 1.5262
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 153s 3s/step - accuracy: 0.5522 - loss: 1.5820 - val_accuracy: 0.6164 - val_loss: 1.2214
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 155s 3s/step - accuracy: 0.6486 - loss: 1.2593 - val_accuracy: 0.6610 - val_loss: 1.0685
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.6936 - loss: 1.0697 - val_accuracy: 0.7158 - val_loss: 0.8688
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 155s 3s/step - accuracy: 0.7248 - loss: 0.9179 - val_accuracy: 0.7089 - val_loss: 0.8517
Epoch 7/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.7669 - loss: 0.8111 - val_accuracy: 0.7603 - val_loss: 0.7691
Epoch 8/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 145s 3s/step - accuracy: 0.7740 - loss: 0.7717 - val_accuracy: 0.7500 - v

In [27]:
model.save(MODEL_PATH)

In [28]:
def predict_image(model, img_path, target_size=(IMAGE_SIZE, IMAGE_SIZE), threshold=0.6):
    """
    Returns the predicted class and confidence.
    If confidence < threshold, returns 'Unknown'.
    """
    try:
        img = image.load_img(img_path, target_size=target_size)
        img_array = np.expand_dims(image.img_to_array(img)/255.0, axis=0)
        preds = model.predict(img_array)
        class_idx = np.argmax(preds)
        class_labels = list(train_generator.class_indices.keys())
        confidence = float(np.max(preds))
        if confidence < threshold:
            return "Unknown", confidence
        return class_labels[class_idx], confidence
    except Exception as e:
        return f"Error: {e}", 0.0

In [29]:
img_path = '../trialImages/apple.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction: mango (96.62%)


In [30]:
img_path = '../trialImages/banana.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: banana (69.94%)


In [31]:
img_path = '../trialImages/beetroot.png'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Prediction: beetroot (99.98%)


In [32]:
img_path = '../trialImages/Bell Pepper.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction: Unknown (59.03%)


In [33]:
img_path = '../trialImages/cabbage.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Prediction: cabbage (100.00%)


In [34]:
img_path = '../trialImages/capsicum.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: Unknown (36.82%)


In [35]:
img_path = '../trialImages/carrot.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction: sweetpotato (83.06%)


In [36]:
img_path = '../trialImages/cauliflower.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction: cauliflower (99.99%)


In [37]:
img_path = '../trialImages/chilli peper.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction: chilli pepper (100.00%)


In [38]:
img_path = '../trialImages/corn.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Prediction: Unknown (55.08%)


In [39]:
img_path = '../trialImages/cucumber.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: cucumber (100.00%)


In [40]:
img_path = '../trialImages/egg plant.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: eggplant (99.99%)


In [41]:
img_path = '../trialImages/garlic.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: garlic (96.80%)


In [42]:
img_path = '../trialImages/ginger.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: ginger (99.04%)


In [43]:
img_path = '../trialImages/grapes.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: grapes (94.92%)


In [44]:
img_path = '../trialImages/jalapeno.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: jalepeno (84.18%)


In [45]:
img_path = '../trialImages/kiwi.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: Unknown (52.39%)


In [46]:
img_path = '../trialImages/lemon.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: mango (99.93%)


In [47]:
img_path = '../trialImages/lettuce.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction: lettuce (99.81%)


In [48]:
img_path = '../trialImages/mango.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: orange (99.23%)


In [49]:
img_path = '../trialImages/onion.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: onion (99.99%)


In [50]:
img_path = '../trialImages/orange.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: Unknown (43.29%)


In [51]:
img_path = '../trialImages/paprika.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: capsicum (87.16%)


In [52]:
img_path = '../trialImages/pear.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction: mango (97.01%)


In [53]:
img_path = '../trialImages/peas.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction: cucumber (95.89%)


In [54]:
img_path = '../trialImages/pineapple.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: pineapple (100.00%)


In [55]:
img_path = '../trialImages/pomogranate.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction: mango (89.72%)


In [56]:
img_path = '../trialImages/potato.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: potato (71.21%)


In [57]:
img_path = '../trialImages/soybeans.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction: soy beans (99.99%)


In [58]:
img_path = '../trialImages/spinach.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: spinach (100.00%)


In [59]:
img_path = '../trialImages/strawberry.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction: raddish (71.25%)


In [60]:
img_path = '../trialImages/sweetcorn.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction: sweetcorn (66.86%)


In [61]:
img_path = '../trialImages/sweetpotato.jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: sweetpotato (89.54%)


In [62]:
img_path = '../trialImages/tomato.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Prediction: tomato (100.00%)


In [63]:
img_path = '../trialImages/turnip.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction: raddish (99.99%)


In [64]:
img_path = '../trialImages/watermelon.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: watermelon (100.00%)


In [65]:
img_path = '../trialImages/image1.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: Unknown (56.96%)


In [66]:
img_path = '../trialImages/image2.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction: banana (83.01%)


In [67]:
img_path = '../trialImages/image3.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction: orange (99.40%)


In [68]:
img_path = '../trialImages/image4.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction: pineapple (97.22%)


In [69]:
img_path = '../trialImages/apple_bulk (1).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: apple (87.55%)


In [70]:
img_path = '../trialImages/apple_bulk (10).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction: Unknown (43.89%)


In [71]:
img_path = '../trialImages/apple_bulk (14).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: Unknown (35.66%)


In [72]:
img_path = '../trialImages/apple_bulk (7).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction: mango (96.55%)


In [73]:
img_path = '../trialImages/banana (11).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: banana (97.16%)


In [74]:
img_path = '../trialImages/banana (22).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: mango (98.22%)


In [75]:
img_path = '../trialImages/banana (8).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: banana (98.76%)


In [76]:
img_path = '../trialImages/beefsteak_tomato (1).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction: tomato (99.62%)


In [77]:
img_path = '../trialImages/beefsteak_tomato (17).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: Unknown (33.37%)


In [78]:
img_path = '../trialImages/beefsteak_tomato (6).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction: tomato (93.50%)


In [79]:
img_path = '../trialImages/carrot (10).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: sweetpotato (67.16%)


In [80]:
img_path = '../trialImages/carrot (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction: carrot (67.33%)


In [81]:
img_path = '../trialImages/carrot (8).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: carrot (98.64%)


In [82]:
img_path = '../trialImages/Image_6.png'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: chilli pepper (100.00%)


In [83]:
img_path = '../trialImages/Image_33.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: mango (100.00%)


In [84]:
img_path = '../trialImages/cucumber (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction: cucumber (99.82%)


In [85]:
img_path = '../trialImages/cucumber (8).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction: cucumber (96.78%)


In [86]:
img_path = '../trialImages/flat_cabbage (4).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: Unknown (49.56%)


In [87]:
img_path = '../trialImages/flat_cabbage (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: spinach (75.13%)


In [88]:
img_path = '../trialImages/flat_cabbage (6).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: Unknown (51.87%)


In [89]:
img_path = '../trialImages/flat_cabbage (7).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction: cabbage (95.08%)


In [90]:
img_path = '../trialImages/grapes (12).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Prediction: sweetpotato (86.86%)


In [91]:
img_path = '../trialImages/grapes (2).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction: Unknown (47.47%)


In [92]:
img_path = '../trialImages/grapes (7).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: Unknown (44.50%)


In [93]:
img_path = '../trialImages/beetroot (1).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Prediction: beetroot (99.10%)


In [94]:
img_path = '../trialImages/beetroot (3).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: sweetpotato (77.81%)


In [95]:
img_path = '../trialImages/beetroot (5).jpeg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: beetroot (93.29%)


In [96]:
img_path = './banana1.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction: banana (99.81%)


In [97]:
img_path = './hello.jpg'  # Replace with your test image
pred_class, confidence = predict_image(model, img_path)
print(f"Prediction: {pred_class} ({confidence*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Prediction: carrot (99.99%)
